# 模型批量验证对比 Notebook

本 Notebook 用于在 **train / valid / test** 三个数据集上系统性对比微调模型与原始模型的检测性能。

整体内容按照“先推理分析、后结果对比”的顺序组织：先完成模型加载、数据集准备、单/多GPU推理与指标汇总，再进入结果展示、可视化对比和差异分析。

多GPU模式默认启用 `rank0` 的 tqdm 总进度条，并通过 `tqdm.write(...)` 输出固定宽度的阶段摘要日志，避免进度条与状态打印相互覆盖。

## 评估维度

1. **数值量化指标**: IOU、精确率(Precision)、召回率(Recall)、F1分数、检测准确率
2. **跨数据集对比**: 在 train / valid / test 上分别评估，观察泛化能力
3. **可视化对比**: 指标雷达图、分组柱状图、差异热力图
4. **差异分析**: 自动识别关键差异点并生成简要分析

---


## 0. 环境配置


In [ ]:
import json
import os
import re
import sys
import time
import warnings
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np

warnings.filterwarnings("ignore")

try:
    from io import BytesIO

    import matplotlib.patches as mpatches
    import matplotlib.pyplot as plt
    import requests
    import torch
    from matplotlib.gridspec import GridSpec
    from PIL import Image, ImageDraw, ImageFont

    if hasattr(__builtins__, "__IPYTHON__"):
        from tqdm.notebook import tqdm
    else:
        from tqdm import tqdm

    import matplotlib.font_manager as fm

    chinese_fonts = ["SimHei", "Microsoft YaHei", "PingFang SC", "Heiti SC", "STHeiti", "WenQuanYi Micro Hei", "Noto Sans CJK SC", "Source Han Sans SC"]
    available_fonts = [f.name for f in fm.fontManager.ttflist]
    font_found = None
    for font_name in chinese_fonts:
        if font_name in available_fonts:
            font_found = font_name
            break
    if font_found:
        plt.rcParams["font.sans-serif"] = [font_found, "DejaVu Sans"]
        plt.rcParams["axes.unicode_minus"] = False
    else:
        print("警告: 未找到中文字体, 图表中文可能显示为方块")
        plt.rcParams["font.sans-serif"] = ["DejaVu Sans"]

    print("依赖加载成功")
    print(f"PyTorch版本: {torch.__version__}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"GPU内存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
except ImportError as e:
    print(f"缺少依赖: {e}")
    print("请安装: pip install torch pillow matplotlib requests numpy")
    sys.exit(1)

依赖加载成功
PyTorch版本: 2.10.0+cu128
GPU: NVIDIA RTX 5880 Ada Generation
GPU内存: 47.4 GB


## 1. 评估指标模块

包含 IOU 计算器（边界框重叠度）和检测指标计算器（精确率/召回率/F1等）。


In [2]:
class IOUCalculator:
    """IOU(交并比)计算器"""

    @staticmethod
    def calculate_iou(box1: List[float], box2: List[float]) -> float:
        """计算两个边界框的IOU, bbox格式: [x1, y1, x2, y2]"""
        x1_1, y1_1, x2_1, y2_1 = box1
        x1_2, y1_2, x2_2, y2_2 = box2

        xi1 = max(x1_1, x1_2)
        yi1 = max(y1_1, y1_2)
        xi2 = min(x2_1, x2_2)
        yi2 = min(y2_1, y2_2)

        if xi2 <= xi1 or yi2 <= yi1:
            return 0.0

        inter_area = (xi2 - xi1) * (yi2 - yi1)
        box1_area = (x2_1 - x1_1) * (y2_1 - y1_1)
        box2_area = (x2_2 - x1_2) * (y2_2 - y1_2)
        union_area = box1_area + box2_area - inter_area

        return inter_area / union_area if union_area > 0 else 0.0

    @staticmethod
    def calculate_batch_iou(detections1: List[Dict], detections2: List[Dict]) -> Dict[str, Any]:
        """批量计算两组检测结果的IOU"""
        if not detections1 or not detections2:
            return {
                "mean_iou": 0.0,
                "max_iou": 0.0,
                "min_iou": 0.0,
                "num_pairs": 0,
                "class_ious": {},
            }

        all_ious = []
        class_ious = {}

        for det1 in detections1:
            bbox1 = det1.get("bbox", [0, 0, 0, 0])
            label1 = det1.get("label", "unknown")

            best_iou = 0.0
            for det2 in detections2:
                bbox2 = det2.get("bbox", [0, 0, 0, 0])
                iou = IOUCalculator.calculate_iou(bbox1, bbox2)
                best_iou = max(best_iou, iou)

            if best_iou > 0:
                all_ious.append(best_iou)
                if label1 not in class_ious:
                    class_ious[label1] = []
                class_ious[label1].append(best_iou)

        class_stats = {}
        for label, ious in class_ious.items():
            class_stats[label] = {
                "mean_iou": np.mean(ious),
                "max_iou": np.max(ious),
                "min_iou": np.min(ious),
                "count": len(ious),
            }

        return {
            "mean_iou": float(np.mean(all_ious)) if all_ious else 0.0,
            "max_iou": float(np.max(all_ious)) if all_ious else 0.0,
            "min_iou": float(np.min(all_ious)) if all_ious else 0.0,
            "num_pairs": len(all_ious),
            "class_ious": class_stats,
        }

    @staticmethod
    def filter_by_confidence(detections: List[Dict], threshold: float) -> List[Dict]:
        """根据置信度阈值过滤检测结果"""
        return [det for det in detections if det.get("confidence", 0) >= threshold]


class MetricsCalculator:
    """检测评估指标计算器

    通过IOU匹配计算精确率、召回率、F1等指标。
    匹配策略: 对每个检测结果找最佳IOU的ground truth, IOU >= threshold 则视为匹配。
    """

    DEFAULT_IOU_THRESHOLD = 0.5

    @staticmethod
    def compute_sample_metrics(
        detections: List[Dict],
        ground_truth: List[Dict],
        iou_threshold: float = 0.5,
    ) -> Dict[str, Any]:
        """计算单样本的检测指标"""
        if not ground_truth:
            if not detections:
                return {
                    "precision": 1.0,
                    "recall": 1.0,
                    "f1": 1.0,
                    "num_det": 0,
                    "num_gt": 0,
                    "num_match": 0,
                    "mean_match_iou": 0.0,
                    "det_success": True,
                }
            return {
                "precision": 0.0,
                "recall": 1.0,
                "f1": 0.0,
                "num_det": len(detections),
                "num_gt": 0,
                "num_match": 0,
                "mean_match_iou": 0.0,
                "det_success": False,
            }

        if not detections:
            return {
                "precision": 1.0,
                "recall": 0.0,
                "f1": 0.0,
                "num_det": 0,
                "num_gt": len(ground_truth),
                "num_match": 0,
                "mean_match_iou": 0.0,
                "det_success": False,
            }

        matched_gt = set()
        matched_det = set()
        match_ious = []

        for i, det in enumerate(detections):
            det_bbox = det.get("bbox", [0, 0, 0, 0])
            best_iou = 0.0
            best_j = -1
            for j, gt in enumerate(ground_truth):
                if j in matched_gt:
                    continue
                gt_bbox = gt.get("bbox", [0, 0, 0, 0])
                iou = IOUCalculator.calculate_iou(det_bbox, gt_bbox)
                if iou > best_iou:
                    best_iou = iou
                    best_j = j
            if best_iou >= iou_threshold and best_j >= 0:
                matched_gt.add(best_j)
                matched_det.add(i)
                match_ious.append(best_iou)

        precision = len(matched_det) / len(detections)
        recall = len(matched_gt) / len(ground_truth)
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        det_success = len(matched_gt) == len(ground_truth) and len(matched_det) == len(detections)

        return {
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "num_det": len(detections),
            "num_gt": len(ground_truth),
            "num_match": len(matched_det),
            "mean_match_iou": float(np.mean(match_ious)) if match_ious else 0.0,
            "det_success": det_success,
        }

    @staticmethod
    def aggregate_metrics(sample_metrics_list: List[Dict]) -> Dict[str, Any]:
        """汇总多个样本的指标, 计算均值和标准差"""
        if not sample_metrics_list:
            return {}

        keys = ["precision", "recall", "f1", "num_det", "num_gt", "num_match", "mean_match_iou"]
        result = {}
        for key in keys:
            values = [m[key] for m in sample_metrics_list if key in m]
            if values:
                result[f"mean_{key}"] = float(np.mean(values))
                result[f"std_{key}"] = float(np.std(values))
            else:
                result[f"mean_{key}"] = 0.0
                result[f"std_{key}"] = 0.0

        result["total_samples"] = len(sample_metrics_list)
        result["success_rate"] = float(np.mean([m.get("det_success", False) for m in sample_metrics_list]))

        # 计算宏平均(macro-average): 先按类别分组计算, 再取平均
        return result


print("IOU计算器和指标计算器已初始化")

IOU计算器和指标计算器已初始化


## 2. 配置参数

修改以下配置单元格中的参数即可适配不同环境。所有路径基于 `PROJECT_ROOT` 自动推导。

**评估模式说明:**

| 模式        | 说明          | 适用场景           |
| ----------- | ------------- | ------------------ |
| `single`    | 单GPU推理评估 | 快速验证, 调试模式 |
| `multi_gpu` | 多GPU并行推理 | 大规模数据集评估   |

**跨Notebook参数关联:**

- `BASE_MODEL_PATH`: 与 **02-微调** 和 **03-推理** Notebook 中的基础模型路径保持一致
- `LORA_ADAPTER_PATH`: 由 **02-微调** Notebook 生成的 LoRA 适配器路径
- `DATA_DIR`: 由 **01-数据预处理** Notebook 生成的 JSONL 数据目录


In [ ]:
from pathlib import Path
import sys

_nb_file = globals().get("__vsc_ipynb_file__", "")
_bootstrap_starts = [Path.cwd()] + ([Path(_nb_file).parent] if _nb_file else [])
for _start in _bootstrap_starts:
    for _candidate in [_start] + list(_start.parents):
        if (_candidate / "pyproject.toml").exists():
            if str(_candidate) not in sys.path:
                sys.path.insert(0, str(_candidate))
            break

from notebooks.common import initialize_notebook_context

NOTEBOOK_CONTEXT = initialize_notebook_context(notebook_file=_nb_file, cwd=Path.cwd(), configure_unsloth_cache=True)
NOTEBOOK_DIR = NOTEBOOK_CONTEXT["NOTEBOOK_DIR"]
PROJECT_ROOT = NOTEBOOK_CONTEXT["PROJECT_ROOT"]
UNSLOTH_CACHE_DIR = NOTEBOOK_CONTEXT["UNSLOTH_CACHE_DIR"]
print(f"Unsloth compile cache: {UNSLOTH_CACHE_DIR}")

# ============================================================
# 评估模式选择
# ============================================================
# "single"       → 单GPU推理评估 + 结果展示/指标分析 (Section 7-8 + Section 10-13)
# "multi_gpu"    → 多GPU并行推理 + 结果展示/指标分析 (Section 8-13)
EVAL_MODE = "multi_gpu"

VALID_EVAL_MODES = {"single", "multi_gpu"}
if EVAL_MODE not in VALID_EVAL_MODES:
    raise ValueError(f"EVAL_MODE必须是{VALID_EVAL_MODES}之一, 当前: {EVAL_MODE}")

print(f"评估模式: {EVAL_MODE}")

# ---------- 模型路径配置 ----------
BASE_MODEL_PATH = "/raid5/sh/model/unsloth/gemma-4-E4B-it-unsloth-bnb-4bit"
LORA_ADAPTER_PATH = str(PROJECT_ROOT / "models" / "finetuned" / "gemma4_e4b_lora" / "ddp_8gpu/20260518_075130")

# ---------- 数据集路径配置 ----------
# 由01-数据预处理Notebook生成的JSONL数据目录
DATA_DIR = str(PROJECT_ROOT / "data" / "processed" / "unsloth_training_data-wgang_40")
# 加载三个数据集
data_dir = Path(DATA_DIR)
split_files = {
    # "train": data_dir / "train.jsonl",
    # "valid": data_dir / "val.jsonl",
    "test": data_dir
    / "test.jsonl",
}


# ---------- 模型加载参数 ----------
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True

# ---------- 推理模式配置 ----------
INFERENCE_MODE = "sequential"
INFERENCE_PARALLEL_MODE = "single_gpu" if EVAL_MODE == "single" else "multi_gpu"

# ---------- 多GPU并行推理配置 ----------
MULTI_GPU_ENABLED = EVAL_MODE == "multi_gpu"
MULTI_GPU_IDS = [0, 1, 2, 3, 4, 5, 6, 7]
MULTI_GPU_STRATEGY = "data_parallel"
MULTI_GPU_SCHEDULER_MODE = "dynamic_queue"
MULTI_GPU_LOAD_BALANCE = "round_robin"
MULTI_GPU_RESULT_DIR = str(PROJECT_ROOT / "results" / "multi_gpu_inference")
MULTI_GPU_LABELME_OUTPUT = True

# ---------- Multi-GPU launch configuration ----------
# Follow the training notebook pattern: notebook prepares config only,
# and the actual multi-GPU inference runs in a standalone torchrun script.
MULTI_GPU_EXECUTOR_BACKEND = "torchrun"  # torchrun | process | thread
MULTI_GPU_BATCH_SIZE = 4
MULTI_GPU_AUTO_RUN_TORCHRUN = True
MULTI_GPU_TORCHRUN_PORT = 29510
MULTI_GPU_TORCHRUN_SCRIPT_PATH = str(PROJECT_ROOT / "distributed_training" / "distributed_inference.py")
MULTI_GPU_TORCHRUN_WORKDIR = str(PROJECT_ROOT)
MULTI_GPU_TORCHRUN_USE_BASH = os.name != "nt"  # legacy; get_ipython().system mode ignores this
MULTI_GPU_TORCHRUN_RAISE_ON_ERROR = True
MULTI_GPU_TORCHRUN_PROGRESS_RANKS = "0"
MULTI_GPU_PARALLEL_MODEL_LOADING = False  # legacy; torchrun mode ignores this
MULTI_GPU_MAX_CONCURRENT_MODEL_LOADS = 1  # legacy; torchrun mode ignores this
MULTI_GPU_MODEL_START_STAGGER_SECONDS = 0.0  # legacy; torchrun mode ignores this
MULTI_GPU_WRITE_LABELME_DURING_INFERENCE = False
MULTI_GPU_MODEL_SCHEDULE = "sequential_rounds"  # torchrun worker runs sequential rounds

# 旧版兼容参数
INFERENCE_GPU_IDS = MULTI_GPU_IDS
INFERENCE_MODELS_PER_GPU = 1
INFERENCE_GPU_GROUPS = [[g] for g in MULTI_GPU_IDS]
INFERENCE_DEVICE_MAP_STRATEGY = "cuda:0"

if INFERENCE_PARALLEL_MODE == "single_gpu":
    DEVICE_MAP = {"": INFERENCE_GPU_IDS[0]}
elif INFERENCE_PARALLEL_MODE == "device_map":
    DEVICE_MAP = INFERENCE_DEVICE_MAP_STRATEGY
else:
    DEVICE_MAP = {"": INFERENCE_GPU_IDS[0]}

# ---------- 推理参数 ----------
INFERENCE_MAX_NEW_TOKENS = 512
INFERENCE_TEMPERATURE = 0.7
INFERENCE_TOP_P = 0.9

# ---------- 评估参数 ----------
IOU_MATCH_THRESHOLD = 0.5
CONFIDENCE_THRESHOLDS = [0.5, 0.7, 0.85, 0.95]
MAX_EVAL_SAMPLES = None

# ---------- 可视化参数 ----------
CHART_DPI = 150
CHART_STYLE = "seaborn-v0_8-whitegrid"

# ---------- 结果持久化配置 ----------
INFERENCE_RESULT_DIR = str(PROJECT_ROOT / "results" / "comparison_cache")
INFERENCE_FORCE_RERUN = False

# 模式功能开关 (single / multi_gpu 均包含结果展示与指标分析)
ENABLE_SINGLE_GPU_INFERENCE = EVAL_MODE == "single"
ENABLE_MULTI_GPU_INFERENCE = EVAL_MODE == "multi_gpu"
ENABLE_METRICS_COMPARISON = EVAL_MODE in {"single", "multi_gpu"}
ENABLE_VISUALIZATION = EVAL_MODE in {"single", "multi_gpu"}

print(f"\n{'='*50}")
print(f"评估配置摘要")
print(f"{'='*50}")
print(f"单GPU推理: {'启用' if ENABLE_SINGLE_GPU_INFERENCE else '跳过'}")
print(f"多GPU推理: {'启用' if ENABLE_MULTI_GPU_INFERENCE else '跳过'}")
print(f"结果展示/指标分析: {'启用' if ENABLE_METRICS_COMPARISON else '跳过'}")
print(f"可视化: {'启用' if ENABLE_VISUALIZATION else '跳过'}")

print(f"\n项目根目录: {PROJECT_ROOT}")
print(f"基础模型: {BASE_MODEL_PATH}")
print(f"LoRA适配器: {LORA_ADAPTER_PATH}")
print(f"数据目录: {DATA_DIR}")
print(f"推理模式: {INFERENCE_MODE} ({INFERENCE_PARALLEL_MODE})")
if MULTI_GPU_ENABLED:
    print(f"多GPU并行: 启用, GPU IDs: {MULTI_GPU_IDS}")
    print(f"并行策略: {MULTI_GPU_STRATEGY}, 调度模式: {MULTI_GPU_SCHEDULER_MODE}, 静态分片: {MULTI_GPU_LOAD_BALANCE}")
    print(f"executor_backend={MULTI_GPU_EXECUTOR_BACKEND}, batch_size={MULTI_GPU_BATCH_SIZE}, progress_ranks={MULTI_GPU_TORCHRUN_PROGRESS_RANKS}")
    print("日志输出: rank0显示tqdm总进度条, 摘要日志通过tqdm.write按固定宽度对齐输出")
print(f"结果缓存目录: {INFERENCE_RESULT_DIR}")
# _MODIFIED_MODE_FLOW_V3

## 3. 模型配置

所有模型路径参数已在上方 **2. 配置参数** 单元格中集中定义，`MODEL_CONFIG` 自动读取这些值。


In [4]:
MODEL_CONFIG = {
    "base_model": {
        "name": "原始模型",
        "base_model_path": BASE_MODEL_PATH,
        "max_seq_length": MAX_SEQ_LENGTH,
        "load_in_4bit": LOAD_IN_4BIT,
        "device_map": DEVICE_MAP,
    },
    "finetuned_model": {
        "name": "微调模型",
        "base_model_path": BASE_MODEL_PATH,
        "lora_adapter_path": LORA_ADAPTER_PATH,
        "max_seq_length": MAX_SEQ_LENGTH,
        "load_in_4bit": LOAD_IN_4BIT,
        "device_map": DEVICE_MAP,
    },
}

print("配置已加载")
print(f"  基础模型: {BASE_MODEL_PATH}")
print(f"  LoRA适配器: {LORA_ADAPTER_PATH}")
print(f"  数据目录: {DATA_DIR}")

配置已加载
  基础模型: /raid5/sh/model/unsloth/gemma-4-E4B-it-unsloth-bnb-4bit
  LoRA适配器: /raid5/sh/code/vlm-detect/models/finetuned/gemma4_e4b_lora/single/20260514_051604
  数据目录: /raid5/sh/code/vlm-detect/data/processed/unsloth_training_data-wgang_40


## 4. 模型加载模块


In [ ]:
class ModelLoader:
    """视觉模型加载器"""

    def __init__(self, config: Dict[str, Any]):
        self.config = config
        self.model = None
        self.processor = None
        self._is_loaded = False

    def _patch_peft_for_gemma4(self) -> bool:
        """修复PEFT对Gemma4ClippableLinear的支持"""
        try:
            from peft.tuners.lora import model as lora_model

            _original = lora_model.LoraModel._create_new_module

            def _patch(lora_config, adapter_name, target, **kwargs):
                if target.__class__.__name__ == "Gemma4ClippableLinear" and hasattr(target, "linear"):
                    return _original(lora_config, adapter_name, target.linear, **kwargs)
                return _original(lora_config, adapter_name, target, **kwargs)

            lora_model.LoraModel._create_new_module = staticmethod(_patch)
            print("PEFT已patch,支持Gemma4ClippableLinear")
            return True
        except Exception as e:
            print(f"Patch失败: {e}")
            return False

    def load_model(self) -> bool:
        """加载视觉模型"""
        try:
            print(f'正在加载模型: {self.config.get("name", "Unknown")}')
            os.environ["UNSLOTH_DISABLE_STATISTICS"] = "1"

            # 禁用 PyTorch Dynamo，避免 FX 符号追踪冲突
            # 解决: 'Detected that you are using FX to symbolically trace a dynamo-optimized function'
            os.environ["TORCH_COMPILE_DISABLE"] = "1"
            try:
                import torch._dynamo

                torch._dynamo.config.suppress_errors = True
                torch._dynamo.reset()
                print("PyTorch Dynamo 已禁用")
            except Exception:
                pass

            from unsloth import FastVisionModel

            base_path = self.config["base_model_path"]
            lora_path = self.config.get("lora_adapter_path", None)

            self.model, self.processor = FastVisionModel.from_pretrained(
                model_name=base_path,
                max_seq_length=self.config["max_seq_length"],
                load_in_4bit=self.config["load_in_4bit"],
                device_map=self.config["device_map"],
                disable_log_stats=True,
            )

            if lora_path and os.path.exists(lora_path):
                print(f"正在加载 LoRA adapter: {lora_path}")
                self._patch_peft_for_gemma4()
                from peft import PeftModel

                self.model = PeftModel.from_pretrained(self.model, lora_path, is_trainable=False)
                print("LoRA adapter 加载成功")

            self._is_loaded = True
            return True
        except Exception as e:
            print(f"模型加载失败: {e}")
            import traceback

            traceback.print_exc()
            return False

    def unload_model(self):
        """卸载模型并释放显存"""
        if not self._is_loaded:
            print("模型未加载, 无需卸载")
            return

        model_name = self.config.get("name", "Unknown")
        del self.model
        del self.processor
        self.model = None
        self.processor = None
        self._is_loaded = False

        import gc

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()

        print(f"模型 [{model_name}] 已卸载, 显存已释放")
        if torch.cuda.is_available():
            for gpu_id in range(torch.cuda.device_count()):
                allocated = torch.cuda.memory_allocated(gpu_id) / 1024**3
                reserved = torch.cuda.memory_reserved(gpu_id) / 1024**3
                total = torch.cuda.get_device_properties(gpu_id).total_memory / 1024**3
                available = total - reserved
                print(f"  GPU {gpu_id}: 已分配={allocated:.2f}GB | 已保留={reserved:.2f}GB | 可用={available:.2f}GB")

    def is_loaded(self) -> bool:
        return self._is_loaded

## 5. 目标检测模块


In [ ]:
class ObjectDetector:
    """
    Object detector for Gemma 4 inference.
    """

    def __init__(self, model_loader: ModelLoader):
        self.model_loader = model_loader
        self._results_cache = {}

    def _build_prompt(self, query: str) -> str:
        """Build a constrained detection prompt."""
        prompt = f"""Analyze this image carefully. {query}

If the target is present, return only a JSON array with this schema:
[
  {{"box_2d": [y1, x1, y2, x2], "label": "target", "confidence": 0.95}}
]

Coordinate rules:
- box_2d uses [y1, x1, y2, x2]
- coordinates are in a 1000x1000 space
- y1 < y2 and x1 < x2
- confidence must be between 0.0 and 1.0

If no target is found, return []"""
        return prompt

    def _resolve_model_device(self, model) -> torch.device:
        """Resolve the active device for plain, PEFT, or device_map models."""
        model_device = getattr(model, "device", None)
        if model_device is not None:
            return model_device

        try:
            return next(model.parameters()).device
        except StopIteration:
            return torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def _move_inputs_to_device(self, inputs, device: torch.device):
        """Keep processor outputs compatible with generate(**inputs)."""
        if hasattr(inputs, "to"):
            return inputs.to(device)

        if isinstance(inputs, dict):
            moved = {}
            for key, value in inputs.items():
                moved[key] = value.to(device) if hasattr(value, "to") else value
            return moved

        raise TypeError(f"processor output type is unsupported: {type(inputs)!r}")

    def _prepare_generation_inputs(self, images: List[Image.Image], queries: List[str], padding: bool):
        """Build text prompts first, then call processor(...) explicitly."""
        model = self.model_loader.model
        processor = self.model_loader.processor
        prompts = [self._build_prompt(query) for query in queries]
        messages_batch = [
            [
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "image": image},
                        {"type": "text", "text": prompt},
                    ],
                }
            ]
            for image, prompt in zip(images, prompts)
        ]
        texts = [
            processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            for messages in messages_batch
        ]

        processor_kwargs = {
            "text": texts,
            "images": images,
            "return_tensors": "pt",
        }
        if padding:
            processor_kwargs["padding"] = True

        inputs = processor(**processor_kwargs)
        return self._move_inputs_to_device(inputs, self._resolve_model_device(model))

    def _decode_generated_responses(self, processor, inputs, outputs) -> List[str]:
        """Decode only newly generated tokens to reduce prompt noise."""
        if "attention_mask" in inputs:
            prompt_lengths = inputs["attention_mask"].sum(dim=1).tolist()
        elif "input_ids" in inputs:
            prompt_lengths = [inputs["input_ids"].shape[1]] * inputs["input_ids"].shape[0]
        else:
            prompt_lengths = [0] * outputs.shape[0]

        generated_only = []
        for idx, output in enumerate(outputs):
            start = int(prompt_lengths[idx]) if idx < len(prompt_lengths) else 0
            generated_only.append(output[start:])

        return processor.batch_decode(generated_only, skip_special_tokens=True)

    def detect(self, image: Image.Image, query: str, max_new_tokens: int = 512) -> Dict[str, Any]:
        """Run single-image detection."""
        if not self.model_loader.is_loaded():
            return {"error": "model not loaded", "success": False}

        try:
            model = self.model_loader.model
            processor = self.model_loader.processor
            inputs = self._prepare_generation_inputs([image], [query], padding=False)

            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    use_cache=True,
                    temperature=0.7,
                    top_p=0.9,
                    do_sample=True,
                )

            response = self._decode_generated_responses(processor, inputs, outputs)[0]

            width, height = image.size
            detections = self._parse_response(response, query, width, height)

            return {
                "success": True,
                "raw_response": response,
                "detections": detections,
                "query": query,
            }

        except Exception as e:
            return {"error": str(e), "success": False}

    def detect_batch(
        self,
        images: List[Image.Image],
        queries: List[str],
        max_new_tokens: int = 512,
        batch_size: Optional[int] = None,
    ) -> List[Dict[str, Any]]:
        """Run batched detection with automatic single-sample fallback."""
        if not self.model_loader.is_loaded():
            return [{"error": "model not loaded", "success": False} for _ in images]

        if not images:
            return []

        if len(images) != len(queries):
            raise ValueError(f"images and queries length mismatch: {len(images)} != {len(queries)}")

        model = self.model_loader.model
        processor = self.model_loader.processor
        batch_size = batch_size or len(images)
        results: List[Dict[str, Any]] = []
        for start in range(0, len(images), batch_size):
            image_batch = images[start : start + batch_size]
            query_batch = queries[start : start + batch_size]
            try:
                inputs = self._prepare_generation_inputs(image_batch, query_batch, padding=len(image_batch) > 1)

                with torch.no_grad():
                    outputs = model.generate(
                        **inputs,
                        max_new_tokens=max_new_tokens,
                        use_cache=True,
                        temperature=INFERENCE_TEMPERATURE,
                        top_p=INFERENCE_TOP_P,
                        do_sample=True,
                    )

                responses = self._decode_generated_responses(processor, inputs, outputs)
                for image, query, response in zip(image_batch, query_batch, responses):
                    width, height = image.size
                    detections = self._parse_response(response, query, width, height)
                    results.append(
                        {
                            "success": True,
                            "raw_response": response,
                            "detections": detections,
                            "query": query,
                        }
                    )
            except Exception as batch_error:
                for image, query in zip(image_batch, query_batch):
                    single = self.detect(image, query, max_new_tokens=max_new_tokens)
                    if not single.get("success"):
                        single["error"] = single.get("error") or str(batch_error)
                    results.append(single)

        return results

    def _parse_response(self, response: str, query: str, width: int, height: int) -> List[Dict[str, Any]]:
        """Parse model response and convert box_2d coordinates to pixel boxes."""
        import json
        import re

        detections = []
        scale_1000_x = width / 1000.0
        scale_1000_y = height / 1000.0

        def _is_normalized(coords: list) -> bool:
            return all(0 <= v <= 1 for v in coords)

        def _convert_coords(coords: list, w: int, h: int) -> tuple:
            if _is_normalized(coords):
                x1 = int(coords[1] * w)
                y1 = int(coords[0] * h)
                x2 = int(coords[3] * w)
                y2 = int(coords[2] * h)
            else:
                x1 = int(coords[1] * scale_1000_x)
                y1 = int(coords[0] * scale_1000_y)
                x2 = int(coords[3] * scale_1000_x)
                y2 = int(coords[2] * scale_1000_y)
            return (x1, y1, x2, y2)

        def _sanitize_box(x1: int, y1: int, x2: int, y2: int):
            x1 = max(0, min(x1, width - 1))
            y1 = max(0, min(y1, height - 1))
            x2 = max(0, min(x2, width - 1))
            y2 = max(0, min(y2, height - 1))
            if x2 <= x1 or y2 <= y1:
                return None
            return (x1, y1, x2, y2)

        def _extract_json_array(text: str) -> Optional[str]:
            json_block = re.search(r"```json\s*([\s\S]*?)\s*```", text)
            if json_block:
                return json_block.group(1).strip()

            start_idx = text.find("[")
            if start_idx == -1:
                return None

            bracket_count = 0
            for i, char in enumerate(text[start_idx:], start_idx):
                if char == "[":
                    bracket_count += 1
                elif char == "]":
                    bracket_count -= 1
                    if bracket_count == 0:
                        return text[start_idx : i + 1]
            return None

        def _append_detection(item: dict):
            coords = item.get("box_2d")
            if not isinstance(coords, list) or len(coords) != 4:
                return

            x1, y1, x2, y2 = _convert_coords(coords, width, height)
            sanitized = _sanitize_box(x1, y1, x2, y2)
            if sanitized is None:
                return

            confidence = item.get("confidence", 0.85)
            try:
                confidence = float(confidence)
            except (TypeError, ValueError):
                confidence = 0.85

            detections.append(
                {
                    "bbox": [sanitized[0], sanitized[1], sanitized[2], sanitized[3]],
                    "label": item.get("label", "object"),
                    "confidence": max(0.0, min(confidence, 1.0)),
                }
            )

        json_str = _extract_json_array(response)
        if json_str:
            try:
                json_data = json.loads(json_str)
                if isinstance(json_data, list):
                    for item in json_data:
                        if isinstance(item, dict):
                            _append_detection(item)
                if detections:
                    return detections
            except (json.JSONDecodeError, TypeError, ValueError):
                pass

        obj_pattern = r'\{[^{}]*"box_2d"[^{}]*\}'
        for obj_str in re.findall(obj_pattern, response, re.DOTALL):
            try:
                obj = json.loads(obj_str)
                if isinstance(obj, dict):
                    _append_detection(obj)
            except (json.JSONDecodeError, TypeError, ValueError):
                continue

        return detections


## 6. 可视化模块

包含检测结果可视化对比器和指标可视化器（雷达图、柱状图等）。


In [7]:
class ComparisonVisualizer:
    """检测结果可视化对比器"""

    COLORS = [
        "#FF3838",
        "#FF9D00",
        "#FF701F",
        "#FFB21D",
        "#CFD231",
        "#48F90A",
        "#92CC17",
        "#3DDB86",
        "#1A9F34",
        "#00D4BB",
    ]

    def draw_detections(self, image: Image.Image, detections: List[Dict]) -> Image.Image:
        if not detections:
            return image

        img_draw = image.copy()
        draw = ImageDraw.Draw(img_draw)

        try:
            font = ImageFont.load_default(size=16)
        except Exception:
            font = ImageFont.load_default()

        for i, det in enumerate(detections):
            bbox = det.get("bbox", [0, 0, 0, 0])
            label = det.get("label", "未知")
            confidence = det.get("confidence", 0)

            if len(bbox) == 4:
                x1, y1, x2, y2 = int(bbox[0]), int(bbox[1]), int(bbox[2]), int(bbox[3])
            else:
                continue

            color = self.COLORS[i % len(self.COLORS)]
            draw.rectangle([x1, y1, x2, y2], outline=color, width=3)

            text = f"{label} {confidence:.0%}"
            text_bbox = draw.textbbox((x1, y1), text, font=font)
            text_width = text_bbox[2] - text_bbox[0]
            text_height = text_bbox[3] - text_bbox[1]

            fill_y1 = y1 - text_height - 4
            if fill_y1 < 0:
                fill_y1 = y1 + 4
            fill_y2 = fill_y1 + text_height + 2

            draw.rectangle([x1, fill_y1, x1 + text_width + 4, fill_y2], fill=color)
            draw.text((x1 + 2, fill_y1 + 1), text, fill="white", font=font)

        return img_draw

    def create_comparison_plot(
        self,
        image: Image.Image,
        det_base: List[Dict],
        det_finetuned: List[Dict],
        iou_stats: Dict,
    ):
        fig = plt.figure(figsize=(16, 10))
        gs = GridSpec(2, 2, figure=fig, height_ratios=[3, 1])

        ax1 = fig.add_subplot(gs[0, 0])
        ax2 = fig.add_subplot(gs[0, 1])

        img_base = self.draw_detections(image, det_base)
        img_finetuned = self.draw_detections(image, det_finetuned)

        ax1.imshow(img_base)
        ax1.set_title(f"原始模型\n检测数量: {len(det_base)}", fontsize=12, fontweight="bold")
        ax1.axis("off")

        ax2.imshow(img_finetuned)
        ax2.set_title(f"微调模型\n检测数量: {len(det_finetuned)}", fontsize=12, fontweight="bold")
        ax2.axis("off")

        ax3 = fig.add_subplot(gs[1, :])
        stats_text = "IOU统计:\n"
        stats_text += f'平均IOU: {iou_stats.get("mean_iou", 0):.3f}\n'
        stats_text += f'最大IOU: {iou_stats.get("max_iou", 0):.3f}\n'
        stats_text += f'匹配数量: {iou_stats.get("num_pairs", 0)}\n'
        ax3.text(0.5, 0.5, stats_text, ha="center", va="center", fontsize=11, family="monospace")
        ax3.axis("off")

        plt.tight_layout()
        plt.show()


class MetricsVisualizer:
    """指标可视化器 - 生成跨数据集对比图表"""

    MODEL_COLORS = {"原始模型": "#4C72B0", "微调模型": "#DD8452"}
    SPLIT_COLORS = {"train": "#4C72B0", "valid": "#55A868", "test": "#C44E52"}

    @staticmethod
    def plot_metrics_bar_chart(metrics_dict: Dict, title: str = "模型性能对比"):
        """分组柱状图: 每个指标在不同数据集上两个模型的对比"""
        metric_keys = ["mean_precision", "mean_recall", "mean_f1", "mean_mean_match_iou", "success_rate"]
        metric_labels = ["精确率", "召回率", "F1分数", "平均匹配IOU", "检测成功率"]
        splits = list(metrics_dict.keys())
        models = ["base", "finetuned"]
        model_labels = ["原始模型", "微调模型"]

        n_metrics = len(metric_keys)
        n_splits = len(splits)

        fig, axes = plt.subplots(1, n_splits, figsize=(5 * n_splits, 6), sharey=True)
        if n_splits == 1:
            axes = [axes]

        for ax_idx, split in enumerate(splits):
            ax = axes[ax_idx]
            split_data = metrics_dict[split]
            x = np.arange(n_metrics)
            width = 0.35

            base_vals = []
            ft_vals = []
            for key in metric_keys:
                base_vals.append(split_data.get("base", {}).get(key, 0.0))
                ft_vals.append(split_data.get("finetuned", {}).get(key, 0.0))

            bars1 = ax.bar(x - width / 2, base_vals, width, label="原始模型", color=MetricsVisualizer.MODEL_COLORS["原始模型"], alpha=0.85)
            bars2 = ax.bar(x + width / 2, ft_vals, width, label="微调模型", color=MetricsVisualizer.MODEL_COLORS["微调模型"], alpha=0.85)

            ax.set_title(f"{split} 数据集", fontsize=13, fontweight="bold")
            ax.set_xticks(x)
            ax.set_xticklabels(metric_labels, fontsize=9)
            ax.set_ylim(0, 1.05)
            ax.legend(fontsize=9)
            ax.grid(axis="y", alpha=0.3)

            for bar in bars1:
                h = bar.get_height()
                if h > 0.01:
                    ax.text(bar.get_x() + bar.get_width() / 2, h + 0.02, f"{h:.2f}", ha="center", va="bottom", fontsize=7)
            for bar in bars2:
                h = bar.get_height()
                if h > 0.01:
                    ax.text(bar.get_x() + bar.get_width() / 2, h + 0.02, f"{h:.2f}", ha="center", va="bottom", fontsize=7)

        fig.suptitle(title, fontsize=15, fontweight="bold", y=1.02)
        plt.tight_layout()
        plt.show()

    @staticmethod
    def plot_radar_chart(metrics_dict: Dict, title: str = "模型性能雷达图"):
        """雷达图: 同时展示两个模型在所有指标上的表现"""
        metric_keys = ["mean_precision", "mean_recall", "mean_f1", "mean_mean_match_iou", "success_rate"]
        metric_labels = ["精确率", "召回率", "F1", "匹配IOU", "成功率"]
        splits = list(metrics_dict.keys())

        n_splits = len(splits)
        fig, axes = plt.subplots(1, n_splits, figsize=(6 * n_splits, 6), subplot_kw=dict(polar=True))
        if n_splits == 1:
            axes = [axes]

        angles = np.linspace(0, 2 * np.pi, len(metric_keys), endpoint=False).tolist()
        angles += angles[:1]

        for ax_idx, split in enumerate(splits):
            ax = axes[ax_idx]
            split_data = metrics_dict[split]

            base_vals = [split_data.get("base", {}).get(k, 0.0) for k in metric_keys]
            ft_vals = [split_data.get("finetuned", {}).get(k, 0.0) for k in metric_keys]
            base_vals += base_vals[:1]
            ft_vals += ft_vals[:1]

            ax.plot(angles, base_vals, "o-", linewidth=2, label="原始模型", color=MetricsVisualizer.MODEL_COLORS["原始模型"])
            ax.fill(angles, base_vals, alpha=0.15, color=MetricsVisualizer.MODEL_COLORS["原始模型"])
            ax.plot(angles, ft_vals, "o-", linewidth=2, label="微调模型", color=MetricsVisualizer.MODEL_COLORS["微调模型"])
            ax.fill(angles, ft_vals, alpha=0.15, color=MetricsVisualizer.MODEL_COLORS["微调模型"])

            ax.set_xticks(angles[:-1])
            ax.set_xticklabels(metric_labels, fontsize=10)
            ax.set_ylim(0, 1.0)
            ax.set_title(f"{split} 数据集", fontsize=12, fontweight="bold", pad=20)
            ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=9)

        fig.suptitle(title, fontsize=15, fontweight="bold", y=1.05)
        plt.tight_layout()
        plt.show()

    @staticmethod
    def plot_diff_heatmap(metrics_dict: Dict, title: str = "微调改进幅度热力图"):
        """热力图: 展示微调模型相对于原始模型的改进幅度(差值)"""
        metric_keys = ["mean_precision", "mean_recall", "mean_f1", "mean_mean_match_iou", "success_rate"]
        metric_labels = ["精确率", "召回率", "F1", "匹配IOU", "成功率"]
        splits = list(metrics_dict.keys())

        diff_matrix = np.zeros((len(splits), len(metric_keys)))
        for i, split in enumerate(splits):
            split_data = metrics_dict[split]
            for j, key in enumerate(metric_keys):
                base_val = split_data.get("base", {}).get(key, 0.0)
                ft_val = split_data.get("finetuned", {}).get(key, 0.0)
                diff_matrix[i, j] = ft_val - base_val

        fig, ax = plt.subplots(figsize=(8, 4))
        im = ax.imshow(diff_matrix, cmap="RdYlGn", aspect="auto", vmin=-0.3, vmax=0.3)

        ax.set_xticks(np.arange(len(metric_keys)))
        ax.set_xticklabels(metric_labels, fontsize=10)
        ax.set_yticks(np.arange(len(splits)))
        ax.set_yticklabels(splits, fontsize=10)

        for i in range(len(splits)):
            for j in range(len(metric_keys)):
                val = diff_matrix[i, j]
                color = "white" if abs(val) > 0.15 else "black"
                ax.text(j, i, f"{val:+.3f}", ha="center", va="center", fontsize=9, color=color)

        ax.set_title(title, fontsize=13, fontweight="bold")
        fig.colorbar(im, ax=ax, label="改进幅度(微调-原始)", shrink=0.8)
        plt.tight_layout()
        plt.show()

    @staticmethod
    def plot_iou_distribution(all_results: Dict, title: str = "IOU分布对比"):
        """箱线图: 展示各数据集上两个模型的匹配IOU分布"""
        splits = list(all_results.keys())
        fig, ax = plt.subplots(figsize=(10, 5))

        data_for_box = []
        labels_for_box = []
        positions = []
        pos = 1

        for split in splits:
            split_results = all_results[split]
            base_ious = [r["base_metrics"]["mean_match_iou"] for r in split_results if r["base_metrics"]["mean_match_iou"] > 0]
            ft_ious = [r["ft_metrics"]["mean_match_iou"] for r in split_results if r["ft_metrics"]["mean_match_iou"] > 0]

            data_for_box.append(base_ious if base_ious else [0])
            labels_for_box.append(f"{split}-原始")
            positions.append(pos)
            pos += 1

            data_for_box.append(ft_ious if ft_ious else [0])
            labels_for_box.append(f"{split}-微调")
            positions.append(pos)
            pos += 2

        bp = ax.boxplot(data_for_box, positions=positions, patch_artist=True, widths=0.6)
        for i, patch in enumerate(bp["boxes"]):
            model_type = "原始" if "原始" in labels_for_box[i] else "微调"
            color_key = "原始模型" if model_type == "原始" else "微调模型"
            patch.set_facecolor(MetricsVisualizer.MODEL_COLORS[color_key])
            patch.set_alpha(0.7)

        ax.set_xticklabels(labels_for_box, fontsize=9)
        ax.set_ylabel("匹配IOU", fontsize=11)
        ax.set_title(title, fontsize=13, fontweight="bold")
        ax.grid(axis="y", alpha=0.3)
        plt.tight_layout()
        plt.show()


print("可视化模块已初始化")

可视化模块已初始化


## 7. 加载模型

根据推理模式配置, 支持两种加载策略:

- **顺序化模式** (`INFERENCE_MODE = "sequential"`): 同一时刻仅加载一个模型, 降低显存占用。模型在推理阶段逐个加载→推理→卸载, 结果持久化到磁盘支持缓存复用。
- **并行模式** (`INFERENCE_MODE = "parallel"`): 同时加载两个模型到显存, 保留原始行为, 适合显存充足的场景。

**流程控制说明:**

- 当 `EVAL_MODE = "single"` 时, 本节按配置准备单GPU推理
- 当 `EVAL_MODE = "multi_gpu"` 时, 此步骤跳过, 模型在 Section 9 的 `torchrun` worker 中加载


In [ ]:
visualizer = ComparisonVisualizer()
metrics_visualizer = MetricsVisualizer()

if EVAL_MODE == "single":
    if INFERENCE_MODE == "sequential":
        print("=" * 60)
        print("单GPU顺序化推理模式")
        print("=" * 60)
        print(f"推理模式: {INFERENCE_MODE}")
        print(f"并行策略: {INFERENCE_PARALLEL_MODE}")
        print(f"GPU IDs: {INFERENCE_GPU_IDS}")
        print(f"DEVICE_MAP: {DEVICE_MAP}")
        print("模型将在推理阶段逐个加载, 不在此处预加载")
        model_loader_base = None
        model_loader_finetuned = None
        detector_base = None
        detector_finetuned = None
    elif INFERENCE_MODE == "parallel":
        print("=" * 60)
        print("单GPU并行推理模式 - 同时加载两个模型")
        print("=" * 60)
        model_loader_base = ModelLoader(MODEL_CONFIG["base_model"])
        model_loader_finetuned = ModelLoader(MODEL_CONFIG["finetuned_model"])

        print("\n" + "=" * 60)
        print("加载原始模型")
        print("=" * 60)
        success_base = model_loader_base.load_model()

        print("\n" + "=" * 60)
        print("加载微调模型")
        print("=" * 60)
        success_finetuned = model_loader_finetuned.load_model()

        if success_base and success_finetuned:
            print("\n所有模型加载成功!")
            detector_base = ObjectDetector(model_loader_base)
            detector_finetuned = ObjectDetector(model_loader_finetuned)
        else:
            raise RuntimeError("\n模型加载失败,请检查配置")
    else:
        raise ValueError(f"未知推理模式: {INFERENCE_MODE}")
elif EVAL_MODE == "multi_gpu":
    print("=" * 60)
    print("多GPU并行推理模式 - 模型在 Section 9 加载")
    print("=" * 60)
    print("此处跳过预加载, 等待多GPU推理引擎初始化")
    model_loader_base = None
    model_loader_finetuned = None
    detector_base = None
    detector_finetuned = None
else:
    raise ValueError(f"未知 EVAL_MODE: {EVAL_MODE}")


## 8. 数据集加载与解析

从 JSONL 文件加载 train/valid/test 数据集，解析每条记录的图片路径、查询文本和 ground truth 边界框。

**JSONL记录格式:**

```json
{"messages": [...], "images": ["path/to/img"], "metadata": {"image_width": ..., ...}}
```

**Ground truth解析:** 从 assistant 消息中提取边界框坐标，格式为 `[x_min, y_min, x_max, y_max]`。


In [9]:
class DatasetLoader:
    """从JSONL文件加载评估数据集并解析ground truth"""

    @staticmethod
    def load_jsonl(filepath: str) -> List[Dict]:
        """加载JSONL文件, 返回记录列表"""
        records = []
        with open(filepath, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    try:
                        records.append(json.loads(line))
                    except json.JSONDecodeError:
                        continue
        return records

    @staticmethod
    def parse_ground_truth(record: Dict) -> List[Dict]:
        """从JSONL记录的assistant消息中解析ground truth边界框

        支持两种坐标格式:
        - 归一化坐标 (0-1范围): 自动乘以图片宽高转换为像素坐标
        - 像素坐标: 直接使用
        """
        metadata = record.get("metadata", {})
        img_width = metadata.get("image_width", 1000)
        img_height = metadata.get("image_height", 1000)

        messages = record.get("messages", [])
        assistant_text = ""
        for msg in messages:
            if msg.get("role") == "assistant":
                content = msg.get("content", [])
                for item in content:
                    if item.get("type") == "text":
                        assistant_text += item.get("text", "")

        if not assistant_text:
            return []

        # 解析格式: - label: [x_min, y_min, x_max, y_max]
        pattern = r"-\s*(\S+)\s*:\s*\[\s*([\d.]+)\s*,\s*([\d.]+)\s*,\s*([\d.]+)\s*,\s*([\d.]+)\s*\]"
        gt_bboxes = []

        for match in re.finditer(pattern, assistant_text):
            label = match.group(1)
            coords = [float(match.group(i)) for i in range(2, 6)]
            x_min, y_min, x_max, y_max = coords

            if all(0 <= c <= 1 for c in coords):
                x_min_px = int(x_min * img_width)
                y_min_px = int(y_min * img_height)
                x_max_px = int(x_max * img_width)
                y_max_px = int(y_max * img_height)
            else:
                x_min_px = int(x_min)
                y_min_px = int(y_min)
                x_max_px = int(x_max)
                y_max_px = int(y_max)

            gt_bboxes.append(
                {
                    "bbox": [x_min_px, y_min_px, x_max_px, y_max_px],
                    "label": label,
                    "confidence": 1.0,
                }
            )

        return gt_bboxes

    @staticmethod
    def extract_query(record: Dict) -> str:
        """从JSONL记录的user消息中提取查询文本"""
        messages = record.get("messages", [])
        for msg in messages:
            if msg.get("role") == "user":
                content = msg.get("content", [])
                for item in content:
                    if item.get("type") == "text":
                        return item.get("text", "")
        return ""

    @staticmethod
    def extract_image_path(record: Dict) -> str:
        """从JSONL记录中提取图片路径"""
        images = record.get("images", [])
        if images:
            return images[0]
        metadata = record.get("metadata", {})
        return metadata.get("json_path", "")

    @staticmethod
    def load_image(image_path: str) -> Optional[Image.Image]:
        """加载图片, 支持本地路径和URL"""
        try:
            if image_path.startswith("http://") or image_path.startswith("https://"):
                response = requests.get(image_path, timeout=30)
                response.raise_for_status()
                return Image.open(BytesIO(response.content)).convert("RGB")
            else:
                path = Path(image_path)
                if path.exists():
                    return Image.open(path).convert("RGB")
                return None
        except Exception:
            return None


datasets = {}
for split_name, filepath in split_files.items():
    if filepath.exists():
        records = DatasetLoader.load_jsonl(str(filepath))
        datasets[split_name] = records
        print(f"{split_name} 数据集: {len(records)} 条记录 ({filepath})")
    else:
        print(f"警告: {split_name} 数据文件不存在: {filepath}")
        datasets[split_name] = []

total_records = sum(len(r) for r in datasets.values())
print(f"\n共加载 {total_records} 条记录")
if total_records == 0:
    print("错误: 未加载任何数据, 请检查 DATA_DIR 配置")

test 数据集: 181 条记录 (/raid5/sh/code/vlm-detect/data/processed/unsloth_training_data-wgang_40/test.jsonl)

共加载 181 条记录


In [ ]:
import gc
import json as _json_mod
from pathlib import Path as _PathMod

MODIFIED_FLAG_MARKER = "SEQ_PARALLEL_V2_LOADED"


class ResultManager:
    """推理结果持久化管理器"""

    def __init__(self, result_dir: str):
        self.result_dir = _PathMod(result_dir)
        self.result_dir.mkdir(parents=True, exist_ok=True)

    def _model_cache_dir(self, model_key: str) -> _PathMod:
        d = self.result_dir / model_key
        d.mkdir(parents=True, exist_ok=True)
        return d

    def save_sample_results(self, model_key: str, split_name: str, results: list) -> None:
        path = self._model_cache_dir(model_key) / f"{split_name}_samples.json"
        with open(path, "w", encoding="utf-8") as f:
            _json_mod.dump(results, f, ensure_ascii=False, indent=2)
        print(f"已保存 {model_key}/{split_name} 样本结果: {path}")

    def save_aggregated_metrics(self, model_key: str, split_name: str, metrics: dict) -> None:
        path = self._model_cache_dir(model_key) / f"{split_name}_aggregated.json"
        with open(path, "w", encoding="utf-8") as f:
            _json_mod.dump(metrics, f, ensure_ascii=False, indent=2)
        print(f"已保存 {model_key}/{split_name} 汇总指标: {path}")

    def load_sample_results(self, model_key: str, split_name: str) -> list:
        path = self._model_cache_dir(model_key) / f"{split_name}_samples.json"
        if not path.exists():
            return None
        with open(path, "r", encoding="utf-8") as f:
            return _json_mod.load(f)

    def load_aggregated_metrics(self, model_key: str, split_name: str) -> dict:
        path = self._model_cache_dir(model_key) / f"{split_name}_aggregated.json"
        if not path.exists():
            return None
        with open(path, "r", encoding="utf-8") as f:
            return _json_mod.load(f)

    def has_cached_results(self, model_key: str, split_name: str) -> bool:
        samples_path = self._model_cache_dir(model_key) / f"{split_name}_samples.json"
        agg_path = self._model_cache_dir(model_key) / f"{split_name}_aggregated.json"
        return samples_path.exists() and agg_path.exists()

    def clear_cache(self, model_key: str = None, split_name: str = None) -> None:
        if model_key:
            if split_name:
                for suffix in ["_samples.json", "_aggregated.json"]:
                    p = self._model_cache_dir(model_key) / f"{split_name}{suffix}"
                    if p.exists():
                        p.unlink()
            else:
                d = self._model_cache_dir(model_key)
                for p in d.glob("*.json"):
                    p.unlink()
        else:
            for p in self.result_dir.glob("**/*.json"):
                p.unlink()


result_manager = ResultManager(INFERENCE_RESULT_DIR)
print(f"ResultManager已初始化, 缓存目录: {INFERENCE_RESULT_DIR}")

## 9. 多GPU分布式推理

当前 Notebook 推荐使用 `torchrun + distributed_inference.py` 的数据并行方案。该节位于所有结果对比板块之前, 先完成分布式推理与结果落盘, 再进入统一的指标汇总与展示分析流程。

| 方案 | 适用场景 | 优势 | 说明 |
| ---- | -------- | ---- | ---- |
| `torchrun` 数据并行 | 多GPU, 每GPU一个模型实例 | 稳定可靠, 可分片处理大规模评估数据 | Notebook 负责生成命令并在完成后回收结果做展示/分析 |

> **流程控制说明:**
>
> - 当 `EVAL_MODE = "single"` 时, 此步骤跳过
> - 当 `EVAL_MODE = "multi_gpu"` 时, 启用多GPU推理, 完成后继续执行 Section 10-13 的批量评估、结果展示与指标分析
> - `torchrun` 模式会先展示启动命令、关键参数和执行方式, 再按配置决定是否在 Notebook 内直接启动
> - 如果 Notebook 内执行受限, 可直接复制同一条命令到终端运行


In [ ]:
import shlex
from typing import Any


class DistributedInferenceConfig:
    """Generate a torchrun-based multi-GPU inference launch config."""

    VALID_MODES = ("single_gpu", "ddp", "ddp_2x", "device_map", "multi_gpu")

    def __init__(
        self,
        mode: str,
        gpu_ids: list,
        models_per_gpu: int = 1,
        gpu_groups: list = None,
        device_map_strategy: str = "balanced",
        model_configs: dict = None,
        result_dir: str = None,
        data_dir: str = None,
        split_files: dict = None,
        iou_threshold: float = 0.5,
        max_eval_samples: int = None,
        worker_script_path: str = None,
        torchrun_port: int = 29510,
        batch_size: int = 4,
        max_new_tokens: int = 512,
        temperature: float = 0.7,
        top_p: float = 0.9,
        export_labelme: bool = False,
        scheduler_mode: str = "dynamic_queue",
        partition_strategy: str = "round_robin",
    ):
        if mode not in self.VALID_MODES:
            raise ValueError(f"无效模式: {mode}, 有效值: {self.VALID_MODES}")
        self.mode = mode
        self.gpu_ids = gpu_ids
        self.models_per_gpu = models_per_gpu
        self.gpu_groups = gpu_groups or [[g] for g in gpu_ids]
        self.device_map_strategy = device_map_strategy
        self.model_configs = model_configs or {}
        self.result_dir = result_dir or "./results/distributed_cache"
        self.data_dir = data_dir or "./data/processed"
        self.split_files = split_files or {"test": "test.jsonl"}
        self.iou_threshold = iou_threshold
        self.max_eval_samples = max_eval_samples
        self.worker_script_path = worker_script_path or "distributed_training/distributed_inference.py"
        self.torchrun_port = int(torchrun_port)
        self.batch_size = int(batch_size)
        self.max_new_tokens = int(max_new_tokens)
        self.temperature = float(temperature)
        self.top_p = float(top_p)
        self.export_labelme = bool(export_labelme)
        self.scheduler_mode = str(scheduler_mode)
        self.partition_strategy = str(partition_strategy)

    def _apply_defaults(self) -> dict:
        n_gpus = len(self.gpu_ids)
        if self.mode in ("ddp", "multi_gpu"):
            self.models_per_gpu = 1
            self.gpu_groups = [[g] for g in self.gpu_ids]
        elif self.mode == "ddp_2x":
            self.models_per_gpu = 2
            n_groups = max(1, n_gpus // self.models_per_gpu)
            self.gpu_groups = [self.gpu_ids[i * self.models_per_gpu : (i + 1) * self.models_per_gpu] for i in range(n_groups)]
        return {"n_gpus": n_gpus, "n_groups": len(self.gpu_groups)}

    def ensure_worker_script(self) -> Path:
        path = Path(self.worker_script_path)
        if not path.exists():
            raise FileNotFoundError(f"推理脚本不存在: {path}")
        return path

    def _quote(self, value: Any) -> str:
        return shlex.quote(str(value))

    def generate_torchrun_command(self) -> str:
        info = self._apply_defaults()

        if self.mode not in ("multi_gpu", "ddp"):
            return "echo '当前模式不使用 torchrun 多GPU推理'"

        script_path = self.ensure_worker_script()
        test_path = self.split_files.get("test")
        if not test_path:
            raise ValueError("未找到 test 数据文件路径")

        base_config = self.model_configs.get("base_model", {})
        finetuned_config = self.model_configs.get("finetuned_model", {})

        cmd_parts = [
            "torchrun",
            f"--nproc_per_node={info['n_groups']}",
            "--rdzv_backend=c10d",
            f"--rdzv_endpoint=localhost:{self.torchrun_port}",
            self._quote(script_path),
            f"--gpu_ids {','.join(str(g) for g in self.gpu_ids)}",
            f"--base_model_path {self._quote(base_config['base_model_path'])}",
            f"--data_path {self._quote(test_path)}",
            f"--result_dir {self._quote(self.result_dir)}",
            f"--max_seq_length {int(base_config['max_seq_length'])}",
            f"--batch_size {self.batch_size}",
            f"--iou_threshold {self.iou_threshold}",
            f"--max_new_tokens {self.max_new_tokens}",
            f"--temperature {self.temperature}",
            f"--top_p {self.top_p}",
            f"--scheduler_mode {self._quote(self.scheduler_mode)}",
            f"--partition_strategy {self._quote(self.partition_strategy)}",
        ]

        if base_config.get("load_in_4bit"):
            cmd_parts.append("--load_in_4bit")

        lora_path = finetuned_config.get("lora_adapter_path")
        if lora_path:
            cmd_parts.append(f"--lora_adapter_path {self._quote(lora_path)}")

        if self.max_eval_samples:
            cmd_parts.append(f"--max_eval_samples {int(self.max_eval_samples)}")

        if self.export_labelme:
            cmd_parts.append("--export_labelme")

        base_cmd = " ".join(cmd_parts)
        gpu_csv = ",".join(str(g) for g in self.gpu_ids)
        if os.name == "nt":
            return f'set "CUDA_VISIBLE_DEVICES={gpu_csv}" && {base_cmd}'
        return f"CUDA_VISIBLE_DEVICES={gpu_csv} {base_cmd}"

    def format_command_preview(self, command: str) -> str:
        """Render a wrapped torchrun command for notebook display."""
        try:
            parts = shlex.split(command)
        except ValueError:
            return command

        if not parts:
            return command

        if len(parts) == 1:
            return parts[0]

        lines = []
        for index, part in enumerate(parts):
            suffix = " \\" if index < len(parts) - 1 else ""
            if index == 0:
                lines.append(f"{part}{suffix}")
            else:
                lines.append(f"  {part}{suffix}")
        return "\n".join(lines)

    def print_summary(self):
        info = self._apply_defaults()
        test_path = self.split_files.get("test")
        print("=" * 72)
        print("多GPU 分布式推理摘要")
        print("=" * 72)
        print(f"并行模式: {self.mode}")
        print(f"GPU IDs: {self.gpu_ids}")
        print(f"GPU分组: {self.gpu_groups}")
        print(f"torchrun进程数: {info['n_groups']}")
        print(f"worker脚本: {self.worker_script_path}")
        print(f"工作目录: {MULTI_GPU_TORCHRUN_WORKDIR}")
        print(f"rdzv端口: {self.torchrun_port}")
        print(f"测试集: {test_path}")
        print(f"结果目录: {self.result_dir}")
        print(f"batch_size: {self.batch_size}")
        print(f"调度模式: {self.scheduler_mode}")
        print(f"静态分片策略: {self.partition_strategy}")
        print(f"LabelMe导出: {'是' if self.export_labelme else '否'}")
        print(f"自动执行: {'是' if MULTI_GPU_AUTO_RUN_TORCHRUN else '否'}")
        print("=" * 72)


def build_torchrun_env() -> dict:
    env = os.environ.copy()
    pythonpath_entries = [str(PROJECT_ROOT)]
    if env.get("PYTHONPATH"):
        pythonpath_entries.append(env["PYTHONPATH"])
    env["PYTHONPATH"] = os.pathsep.join(pythonpath_entries)
    env.setdefault("GEMMA4_NOTEBOOK_DIR", str(NOTEBOOK_DIR))
    env.setdefault("GEMMA4_LIVE_TQDM_RANKS", MULTI_GPU_TORCHRUN_PROGRESS_RANKS)
    notebook_file = NOTEBOOK_CONTEXT.get("NOTEBOOK_FILE", "")
    if notebook_file:
        env["GEMMA4_NOTEBOOK_FILE"] = notebook_file
    return env


def run_torchrun_in_notebook(ddp_cmd: str, workdir: str, raise_on_error: bool = True) -> int:
    """Run torchrun via get_ipython().system so Notebook preserves terminal-style progress output better."""
    workdir_path = Path(workdir)
    if not workdir_path.exists():
        raise FileNotFoundError(f"工作目录不存在: {workdir_path}")

    env = build_torchrun_env()
    print("\n开始通过 get_ipython().system(...) 执行 torchrun ...")
    print(f"工作目录: {workdir_path}")
    print(f"执行命令: {ddp_cmd}")
    print(f"PYTHONPATH已注入项目根: {PROJECT_ROOT}")
    print(f"日志策略: live_tqdm={MULTI_GPU_TORCHRUN_PROGRESS_RANKS}, rank0显示tqdm总进度条, worker侧摘要日志通过tqdm.write输出")
    print("-" * 72)

    tracked_keys = [
        "PYTHONPATH",
        "GEMMA4_NOTEBOOK_DIR",
        "GEMMA4_NOTEBOOK_FILE",
        "GEMMA4_UNSLOTH_COMPILE_CACHE_DIR",
        "UNSLOTH_COMPILE_LOCATION",
    ]
    old_values = {key: os.environ.get(key) for key in tracked_keys}
    old_cwd = Path.cwd()

    try:
        os.chdir(workdir_path)
        for key, value in env.items():
            if key in tracked_keys and value:
                os.environ[key] = value

        return_code = get_ipython().system(ddp_cmd)
    finally:
        os.chdir(old_cwd)
        for key, old_value in old_values.items():
            if old_value is None:
                os.environ.pop(key, None)
            else:
                os.environ[key] = old_value

    return_code = int(return_code or 0)
    print("-" * 72)
    print(f"torchrun 已结束, return code = {return_code}")
    if return_code != 0 and raise_on_error:
        raise RuntimeError(f"torchrun 执行失败, return code = {return_code}")
    return return_code


if ENABLE_MULTI_GPU_INFERENCE:
    print("=" * 60)
    print("启用多GPU分布式推理")
    print("=" * 60)

    dist_config = DistributedInferenceConfig(
        mode=INFERENCE_PARALLEL_MODE,
        gpu_ids=INFERENCE_GPU_IDS,
        models_per_gpu=INFERENCE_MODELS_PER_GPU,
        gpu_groups=INFERENCE_GPU_GROUPS,
        device_map_strategy=INFERENCE_DEVICE_MAP_STRATEGY,
        model_configs=MODEL_CONFIG,
        result_dir=MULTI_GPU_RESULT_DIR,
        data_dir=DATA_DIR,
        split_files={k: str(v) for k, v in split_files.items()},
        iou_threshold=IOU_MATCH_THRESHOLD,
        max_eval_samples=MAX_EVAL_SAMPLES,
        worker_script_path=MULTI_GPU_TORCHRUN_SCRIPT_PATH,
        torchrun_port=MULTI_GPU_TORCHRUN_PORT,
        batch_size=MULTI_GPU_BATCH_SIZE,
        max_new_tokens=INFERENCE_MAX_NEW_TOKENS,
        temperature=INFERENCE_TEMPERATURE,
        top_p=INFERENCE_TOP_P,
        export_labelme=MULTI_GPU_LABELME_OUTPUT,
        scheduler_mode=MULTI_GPU_SCHEDULER_MODE,
        partition_strategy=MULTI_GPU_LOAD_BALANCE,
    )

    worker_script = dist_config.ensure_worker_script()
    ddp_cmd = dist_config.generate_torchrun_command()
    dist_config.print_summary()

    print("\ntorchrun 启动命令:")
    print("-" * 72)
    print(dist_config.format_command_preview(ddp_cmd))
    print("-" * 72)
    print(f"推理脚本: {worker_script}")
    print("提示: 如 Notebook 内执行受限, 可复制上述命令到终端运行")

    if MULTI_GPU_AUTO_RUN_TORCHRUN:
        run_torchrun_in_notebook(
            ddp_cmd,
            workdir=MULTI_GPU_TORCHRUN_WORKDIR,
            raise_on_error=MULTI_GPU_TORCHRUN_RAISE_ON_ERROR,
        )
    else:
        print("提示: 如需在当前 Notebook 运行, 可调用 run_torchrun_in_notebook(...)")

else:
    print("跳过多GPU分布式推理")
    print(f"当前 EVAL_MODE={EVAL_MODE}, ENABLE_MULTI_GPU_INFERENCE={ENABLE_MULTI_GPU_INFERENCE}")

## 10. 批量推理与指标计算

支持两种评估器: 单GPU 模式直接执行推理, 多GPU 模式则在本节统一回收 Section 9 产出的结果, 生成一致的 `all_results / all_metrics` 结构, 为后续结果对比做好准备。

- **SequentialEvaluator**: 顺序化推理评估器, 同一时刻仅加载一个模型。Phase1先加载微调模型推理并保存结果, 卸载后Phase2加载基础模型推理。结果持久化到磁盘, 支持缓存复用。
- **BatchEvaluator**: 并行评估器, 同时使用两个已加载模型推理, 保留原始行为。

根据 `INFERENCE_MODE` 配置自动选择评估器。


In [ ]:
def infer_multi_gpu_split_name() -> str:
    preferred = ["test", "valid", "train"]
    for name in preferred:
        if name in split_files:
            return name
    if split_files:
        return next(iter(split_files.keys()))
    return "test"


def load_multi_gpu_evaluation_outputs(datasets: Dict, result_dir: str, max_samples: int = None) -> tuple:
    """从 distributed_inference.py 的输出目录恢复 notebook 统一结果结构。"""
    result_dir_path = Path(result_dir)
    ft_path = result_dir_path / "finetuned_results.json"
    base_path = result_dir_path / "base_results.json"
    summary_path = result_dir_path / "comparison_summary.json"

    missing_files = [str(p) for p in [ft_path, base_path] if not p.exists()]
    if missing_files:
        raise FileNotFoundError(f"多GPU结果文件不存在: {missing_files}")

    split_name = infer_multi_gpu_split_name()
    records = datasets.get(split_name, [])
    eval_records = records[:max_samples] if max_samples and max_samples < len(records) else records

    with open(ft_path, "r", encoding="utf-8") as f:
        ft_results = json.load(f)
    with open(base_path, "r", encoding="utf-8") as f:
        base_results = json.load(f)
    summary = {}
    if summary_path.exists():
        with open(summary_path, "r", encoding="utf-8") as f:
            summary = json.load(f)

    print("=" * 60)
    print("加载多GPU推理结果")
    print("=" * 60)
    print(f"结果目录: {result_dir_path}")
    print(f"目标数据集: {split_name}")
    print(f"微调结果数: {len(ft_results)} | 原始结果数: {len(base_results)} | 数据集记录数: {len(eval_records)}")

    def build_buckets(results: List[Dict]) -> Dict:
        buckets = {}
        for item in results:
            key = (item.get("image_path", ""), item.get("query", ""))
            buckets.setdefault(key, []).append(item)
        return buckets

    ft_buckets = build_buckets(ft_results)
    base_buckets = build_buckets(base_results)

    combined_results = []
    missing_pairs = 0
    for idx, record in enumerate(eval_records):
        image_path = DatasetLoader.extract_image_path(record)
        query = DatasetLoader.extract_query(record)
        key = (image_path, query)

        ft_candidates = ft_buckets.get(key, [])
        base_candidates = base_buckets.get(key, [])
        if not ft_candidates or not base_candidates:
            missing_pairs += 1
            continue

        ft_r = ft_candidates.pop(0)
        base_r = base_candidates.pop(0)
        det_base = base_r.get("detections", [])
        det_ft = ft_r.get("detections", [])

        combined_results.append(
            {
                "index": idx,
                "split": split_name,
                "image_path": image_path,
                "query": query,
                "ground_truth": DatasetLoader.parse_ground_truth(record),
                "det_base": det_base,
                "det_ft": det_ft,
                "base_metrics": base_r.get("metrics", {}),
                "ft_metrics": ft_r.get("metrics", {}),
                "model_iou": IOUCalculator.calculate_batch_iou(det_base, det_ft),
            }
        )

    leftover_ft = sum(len(items) for items in ft_buckets.values())
    leftover_base = sum(len(items) for items in base_buckets.values())

    base_metric_list = [item["base_metrics"] for item in combined_results]
    ft_metric_list = [item["ft_metrics"] for item in combined_results]
    base_agg = summary.get("base") or MetricsCalculator.aggregate_metrics(base_metric_list)
    ft_agg = summary.get("finetuned") or MetricsCalculator.aggregate_metrics(ft_metric_list)

    gpu_stats = summary.get("gpu_stats", {})
    if gpu_stats:
        print("\nGPU统计摘要:")
        for worker_name, stats in sorted(gpu_stats.items()):
            print(
                f"  - {worker_name}: processed={stats.get('processed', 0)}, "
                f"failed={stats.get('failed', 0)}, throughput={stats.get('throughput_samples_per_second', 0):.3f} sample/s"
            )

    report_files = summary.get("load_balance_report_files", {})
    if report_files:
        print("\n负载均衡报告:")
        for name, report_path in sorted(report_files.items()):
            print(f"  - {name}: {report_path}")

    load_balance_reports = summary.get("load_balance_reports", {})
    if load_balance_reports:
        print("\n调度性能对比:")
        for model_name, report in sorted(load_balance_reports.items()):
            print(
                f"  - {model_name}: scheduler={report.get('scheduler_mode', 'unknown')}, "
                f"preferred_static={report.get('preferred_static_baseline', 'n/a')}, "
                f"delta={report.get('makespan_delta_seconds_vs_static', 0):.3f}s, "
                f"improvement={report.get('improvement_pct_vs_static', 0):.2f}%"
            )

    if missing_pairs or leftover_ft or leftover_base:
        print(
            "\n警告: 多GPU结果与数据集对齐存在差异, "
            f"missing_pairs={missing_pairs}, leftover_ft={leftover_ft}, leftover_base={leftover_base}"
        )

    return {split_name: combined_results}, {split_name: {"base": base_agg, "finetuned": ft_agg}}


# 评估执行代码
if EVAL_MODE == "single":
    if INFERENCE_MODE == "sequential":
        print("=" * 60)
        print("使用 SequentialEvaluator (顺序化推理)")
        print("=" * 60)
        seq_evaluator = SequentialEvaluator(IOU_MATCH_THRESHOLD)
        all_results, all_metrics = seq_evaluator.evaluate_all(datasets, max_samples=MAX_EVAL_SAMPLES)
    elif INFERENCE_MODE == "parallel":
        print("=" * 60)
        print("使用 BatchEvaluator (并行推理)")
        print("=" * 60)
        evaluator = BatchEvaluator(detector_base, detector_finetuned, IOU_MATCH_THRESHOLD)
        all_results = {}
        all_metrics = {}
        for split_name, records in datasets.items():
            if not records:
                print(f"跳过空数据集: {split_name}")
                continue
            sample_results, aggregated = evaluator.evaluate_dataset(records, split_name, max_samples=MAX_EVAL_SAMPLES)
            all_results[split_name] = sample_results
            all_metrics[split_name] = aggregated
    else:
        raise ValueError(f"未知推理模式: {INFERENCE_MODE}")

    print("\n" + "=" * 60)
    print("单GPU评估完成!")
    print("=" * 60)
elif EVAL_MODE == "multi_gpu":
    print("=" * 60)
    print("多GPU并行推理模式 - 从 Section 9 结果恢复评估数据")
    print("=" * 60)
    all_results, all_metrics = load_multi_gpu_evaluation_outputs(
        datasets,
        result_dir=MULTI_GPU_RESULT_DIR,
        max_samples=MAX_EVAL_SAMPLES,
    )

    print("\n" + "=" * 60)
    print("多GPU评估结果已接入展示/指标分析流程!")
    print("=" * 60)
else:
    raise ValueError(f"未知 EVAL_MODE: {EVAL_MODE}")


## 11. 结果展示

以表格形式展示两个模型在各数据集上的关键指标对比。


In [ ]:
def _display_width(s: str) -> int:
    """计算字符串显示宽度(中文字符宽度为2)"""
    width = 0
    for ch in s:
        width += 2 if "\u4e00" <= ch <= "\u9fff" else 1
    return width


def _pad_to_width(s: str, target_width: int, align: str = "left") -> str:
    """按显示宽度填充字符串到目标宽度"""
    current = _display_width(s)
    padding = target_width - current
    if padding <= 0:
        return s[: target_width // 2 if any("\u4e00" <= c <= "\u9fff" for c in s) else target_width]
    if align == "right":
        return " " * padding + s
    if align == "center":
        left_pad = padding // 2
        return " " * left_pad + s + " " * (padding - left_pad)
    return s + " " * padding


def format_metrics_table(all_metrics: Dict) -> str:
    """格式化指标为文本表格(中英文对齐)"""
    col_widths = {"split": 8, "model": 8, "precision": 10, "recall": 10, "f1": 8, "iou": 10, "success": 10, "samples": 8}
    header_parts = [
        _pad_to_width("数据集", col_widths["split"]),
        _pad_to_width("模型", col_widths["model"]),
        _pad_to_width("精确率", col_widths["precision"]),
        _pad_to_width("召回率", col_widths["recall"]),
        _pad_to_width("F1", col_widths["f1"]),
        _pad_to_width("匹配IOU", col_widths["iou"]),
        _pad_to_width("成功率", col_widths["success"]),
        _pad_to_width("样本数", col_widths["samples"]),
    ]
    header = " | ".join(header_parts)
    total_width = len(header)
    sep = "-" * total_width
    double_sep = "=" * total_width
    lines = [double_sep, header, sep]

    for split_name in ["train", "valid", "test"]:
        if split_name not in all_metrics:
            continue
        split_data = all_metrics[split_name]
        for model_key, model_label in [("base", "原始"), ("finetuned", "微调")]:
            m = split_data.get(model_key, {})
            line = f"{split_name:<8} | {model_label:<8} | {m.get('mean_precision', 0):.3f}    | {m.get('mean_recall', 0):.3f}    | {m.get('mean_f1', 0):.3f}    | {m.get('mean_mean_match_iou', 0):.3f}    | {m.get('success_rate', 0):.3f}    | {m.get('total_samples', 0):<6}"
            lines.append(line)
        lines.append(sep)

    return "\n".join(lines)


def compute_diff_table(all_metrics: Dict) -> str:
    """计算微调模型相对于原始模型的改进幅度"""
    header = f"{'数据集':<8} | {'精确率差':<10} | {'召回率差':<10} | {'F1差':<10} | {'IOU差':<10} | {'成功率差':<10}"
    sep = "-" * len(header)
    lines = ["\n微调模型改进幅度 (微调 - 原始):", sep, header, sep]

    for split_name in ["train", "valid", "test"]:
        if split_name not in all_metrics:
            continue
        base = all_metrics[split_name].get("base", {})
        ft = all_metrics[split_name].get("finetuned", {})

        diffs = {
            "precision": ft.get("mean_precision", 0) - base.get("mean_precision", 0),
            "recall": ft.get("mean_recall", 0) - base.get("mean_recall", 0),
            "f1": ft.get("mean_f1", 0) - base.get("mean_f1", 0),
            "iou": ft.get("mean_mean_match_iou", 0) - base.get("mean_mean_match_iou", 0),
            "success": ft.get("success_rate", 0) - base.get("success_rate", 0),
        }

        line = f"{split_name:<8} | {diffs['precision']:>+8.3f}  | {diffs['recall']:>+8.3f}  | {diffs['f1']:>+8.3f}  | {diffs['iou']:>+8.3f}  | {diffs['success']:>+8.3f}"
        lines.append(line)

    lines.append(sep)
    return "\n".join(lines)


if all_metrics and ENABLE_METRICS_COMPARISON:
    print("=" * 70)
    print("评估结果汇总")
    print("=" * 70)
    print(format_metrics_table(all_metrics))
    print(compute_diff_table(all_metrics))
elif not all_metrics:
    print("暂无评估结果, 请先运行批量评估")
else:
    print(f"当前 EVAL_MODE={EVAL_MODE}, 跳过结果展示 (ENABLE_METRICS_COMPARISON={ENABLE_METRICS_COMPARISON})")

## 12. 可视化对比

生成多维可视化图表，直观展示两个模型在各数据集上的性能差异。

**图表类型:**

- 分组柱状图: 每个数据集上两个模型的各项指标并列对比
- 雷达图: 多维指标同时对比, 一眼看出模型优劣
- 热力图: 微调改进幅度, 绿色=提升, 红色=退化
- IOU分布箱线图: 匹配IOU的分布特征


In [ ]:
if all_metrics and ENABLE_VISUALIZATION:
    try:
        plt.style.use(CHART_STYLE)
    except Exception:
        pass

    metrics_visualizer.plot_metrics_bar_chart(all_metrics, "模型性能分组对比")
    metrics_visualizer.plot_radar_chart(all_metrics, "模型性能雷达图")
    metrics_visualizer.plot_diff_heatmap(all_metrics)

    if all_results:
        metrics_visualizer.plot_iou_distribution(all_results)
elif not all_metrics:
    print("暂无评估结果, 请先运行批量评估")
else:
    print(f"当前 EVAL_MODE={EVAL_MODE}, 跳过可视化 (ENABLE_VISUALIZATION={ENABLE_VISUALIZATION})")

## 13. 差异分析与总结

自动分析微调模型的关键改进点和潜在退化，生成结构化分析报告。


In [ ]:
def generate_analysis(all_metrics: Dict) -> str:
    """自动生成差异分析报告"""
    if not all_metrics:
        return "暂无评估结果"

    report_lines = []
    report_lines.append("=" * 60)
    report_lines.append("微调效果差异分析报告")
    report_lines.append("=" * 60)
    report_lines.append(f"\n生成时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    report_lines.append(f"IOU匹配阈值: {IOU_MATCH_THRESHOLD}")
    report_lines.append("")

    # 1. 总体改进分析
    report_lines.append("一、总体改进分析")
    report_lines.append("-" * 40)

    all_improvements = []
    all_degradations = []

    for split_name in ["train", "valid", "test"]:
        if split_name not in all_metrics:
            continue
        base = all_metrics[split_name].get("base", {})
        ft = all_metrics[split_name].get("finetuned", {})

        metric_names = {
            "mean_precision": "精确率",
            "mean_recall": "召回率",
            "mean_f1": "F1分数",
            "mean_mean_match_iou": "匹配IOU",
            "success_rate": "检测成功率",
        }

        report_lines.append(f"\n  [{split_name} 数据集]")
        for key, label in metric_names.items():
            base_val = base.get(key, 0)
            ft_val = ft.get(key, 0)
            diff = ft_val - base_val
            if diff > 0.01:
                report_lines.append(f"    + {label}: {base_val:.3f} -> {ft_val:.3f} (提升 {diff:+.3f})")
                all_improvements.append((split_name, label, diff))
            elif diff < -0.01:
                report_lines.append(f"    - {label}: {base_val:.3f} -> {ft_val:.3f} (退化 {diff:+.3f})")
                all_degradations.append((split_name, label, diff))
            else:
                report_lines.append(f"    = {label}: {base_val:.3f} -> {ft_val:.3f} (持平)")

    # 2. 关键发现
    report_lines.append("\n\n二、关键发现")
    report_lines.append("-" * 40)

    if all_improvements:
        top_improvements = sorted(all_improvements, key=lambda x: x[2], reverse=True)[:3]
        report_lines.append("\n  最显著改进:")
        for split, label, diff in top_improvements:
            report_lines.append(f"    - {split}数据集的{label}: 提升{diff:+.3f}")

    if all_degradations:
        top_degradations = sorted(all_degradations, key=lambda x: x[2])[:3]
        report_lines.append("\n  需关注退化:")
        for split, label, diff in top_degradations:
            report_lines.append(f"    - {split}数据集的{label}: 退化{diff:+.3f}")

    # 3. 泛化能力分析
    report_lines.append("\n\n三、泛化能力分析")
    report_lines.append("-" * 40)

    splits_available = [s for s in ["train", "valid", "test"] if s in all_metrics]
    if len(splits_available) >= 2:
        train_ft = all_metrics.get("train", {}).get("finetuned", {}).get("mean_f1", 0)
        test_ft = all_metrics.get("test", {}).get("finetuned", {}).get("mean_f1", 0)
        valid_ft = all_metrics.get("valid", {}).get("finetuned", {}).get("mean_f1", 0)

        train_base = all_metrics.get("train", {}).get("base", {}).get("mean_f1", 0)
        test_base = all_metrics.get("test", {}).get("base", {}).get("mean_f1", 0)

        ft_gap = train_ft - test_ft if train_ft and test_ft else 0
        base_gap = train_base - test_base if train_base and test_base else 0

        report_lines.append(f"\n  微调模型 F1: train={train_ft:.3f}, valid={valid_ft:.3f}, test={test_ft:.3f}")
        report_lines.append(f"  原始模型 F1: train={train_base:.3f}, test={test_base:.3f}")
        report_lines.append(f"\n  微调模型 train-test差距: {ft_gap:+.3f}")
        report_lines.append(f"  原始模型 train-test差距: {base_gap:+.3f}")

        if ft_gap < base_gap:
            report_lines.append("\n  结论: 微调模型的泛化差距更小, 泛化能力更优")
        elif ft_gap > base_gap:
            report_lines.append("\n  结论: 微调模型的泛化差距更大, 可能存在过拟合风险")
        else:
            report_lines.append("\n  结论: 两个模型的泛化差距相近")

    # 4. 建议
    report_lines.append("\n\n四、后续优化建议")
    report_lines.append("-" * 40)
    if all_degradations:
        report_lines.append("  - 关注退化指标, 考虑调整训练数据分布或LoRA参数")
    report_lines.append("  - 增加IOU阈值(0.5->0.7)可更严格评估定位精度")
    report_lines.append("  - 可按类别分别统计指标, 找出薄弱类别针对性优化")

    return "\n".join(report_lines)


if all_metrics and ENABLE_METRICS_COMPARISON:
    analysis_report = generate_analysis(all_metrics)
    print(analysis_report)
elif not all_metrics:
    print("暂无评估结果, 无法生成分析报告")
else:
    print(f"当前 EVAL_MODE={EVAL_MODE}, 跳过差异分析 (ENABLE_METRICS_COMPARISON={ENABLE_METRICS_COMPARISON})")

---

## Notebook 完成总结

本 Notebook 按照“先推理分析、后结果对比”的顺序组织内容, 并实现了以下功能:

1. **数据集加载**: 从 JSONL 格式加载 train/valid/test 三个数据集
2. **Ground Truth 解析**: 从训练数据的 assistant 消息中提取边界框坐标
3. **单GPU评估**: 支持 SequentialEvaluator (顺序加载降低显存) 和 BatchEvaluator (并行推理)
4. **多GPU评估**: 通过 `torchrun` 调用 `distributed_inference.py` 执行数据并行推理
5. **结果持久化**: ResultManager 支持单GPU缓存复用, 多GPU结果目录可回收加载
6. **指标计算**: 精确率、召回率、F1、匹配IOU、检测成功率
7. **结果展示**: 文本表格 + 改进幅度表格
8. **可视化对比**: 分组柱状图、雷达图、热力图、IOU分布箱线图
9. **差异分析**: 自动识别关键改进点和退化风险

### 流程控制机制

10. **EVAL_MODE 模式选择**:
    | 模式 | 说明 | 执行流程 |
    |------|------|----------|
    | `single` | 单GPU推理评估 | Section 7-8 + Section 10-13 |
    | `multi_gpu` | 多GPU并行推理评估 | Section 8-13 |

11. **统一后处理链路**:
    - `single` 模式直接产出 `all_results / all_metrics`
    - `multi_gpu` 模式从结果目录恢复统一结构, 继续执行结果展示、可视化和差异分析

12. **异常处理机制**:
    - 数据加载: 文件不存在时警告并跳过
    - 模型加载: 失败时抛出 RuntimeError
    - 推理过程: 单样本失败不影响整体流程
    - 多GPU结果恢复: 缺少结果文件时直接报错, 便于定位问题

13. **模型卸载与显存管理**: ModelLoader.unload_model() 支持显存释放和状态监控

### 使用说明

修改 `EVAL_MODE` 参数即可控制执行流程:

```python
EVAL_MODE = "single"  # 执行单GPU推理评估 + 展示/分析
EVAL_MODE = "multi_gpu"  # 执行多GPU并行推理 + 展示/分析
```

验证过程可重复, 只需修改配置参数并重新运行相关单元即可。
